In [ ]:
!pip install crewai -q
!pip install langchain -q
!py -m pip install -qU langchain-ollama
!pip install -U ollama
!py -m pip install markdown weasyprint -q
!py -m pip install -qU langchain
!py -m pip install --upgrade langchain langchain-core langchain-community
!py -m pip install langchain-classic

In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error,mean_absolute_percentage_error
import numpy as np
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from langchain.tools import tool
import shap
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from pathlib import Path
from pydantic import BaseModel, Field
from typing import Optional
import re

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
import json

In [28]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "llama3.1:8b" 

llm = ChatOllama(
    model=MODEL,
    temperature=0
)

### Code base

In [29]:
HYPERPARAM_REFERENCE = """
VALID HYPERPARAMETERS PER MODEL:

xgb:
  n_estimators:    int, 100-2000   
  learning_rate:   float, 0.001-0.3 
  max_depth:       int, 2-8        
  subsample:       float, 0.5-1.0 
  colsample_bytree:float, 0.5-1.0  
  reg_lambda:      float, 0.1-10.0
  reg_alpha:       float, 0.0-5.0 
  min_child_weight:int, 1-10      

gbr:
  n_estimators:    int, 100-1000
  learning_rate:   float, 0.001-0.2
  max_depth:       int, 2-6
  subsample:       float, 0.5-1.0
  min_samples_leaf:int, 1-20     
  min_samples_split:int, 2-20

svr:
  C:        float, 0.01-1000.0    
  epsilon:  float, 0.0001-1.0     
  gamma:    'scale' or 0.001-10.0 
  kernel:   'rbf' or 'linear'

knn:
  n_neighbors: int, 3-30         
  weights:     'uniform' or 'distance'
  metric:      'euclidean' or 'manhattan'

bayes: no hyperparameters
"""



In [30]:
MODEL_REGISTRY = {
    "xgb":   lambda cfg: XGBRegressor(
                 n_estimators    = cfg.get("n_estimators",    500),
                 learning_rate   = cfg.get("learning_rate",   0.05),
                 max_depth       = cfg.get("max_depth",        3),
                 subsample       = cfg.get("subsample",        0.8),
                 colsample_bytree= cfg.get("colsample_bytree", 0.8),
                 reg_lambda      = cfg.get("reg_lambda",       1.0),
                 reg_alpha       = cfg.get("reg_alpha",        0.0),
                 min_child_weight= cfg.get("min_child_weight", 1),
                 objective="reg:squarederror", verbosity=0,
                 n_jobs=-1, random_state=42),
    "gbr":   lambda cfg: GradientBoostingRegressor(
                 n_estimators    = cfg.get("n_estimators",    300),
                 learning_rate   = cfg.get("learning_rate",   0.05),
                 max_depth       = cfg.get("max_depth",        3),
                 subsample       = cfg.get("subsample",        0.8),
                 min_samples_leaf= cfg.get("min_samples_leaf", 3),
                 min_samples_split=cfg.get("min_samples_split",2),
                 random_state=42),
    "svr":   lambda cfg: SVR(
                 C       = cfg.get("C",       10.0),
                 epsilon = cfg.get("epsilon",  0.01),
                 gamma   = cfg.get("gamma",   "scale"),
                 kernel  = cfg.get("kernel",  "rbf")),
    "bayes": lambda cfg: BayesianRidge(),
    "knn":   lambda cfg: KNeighborsRegressor(
                 n_neighbors = cfg.get("n_neighbors", 7),
                 weights     = cfg.get("weights",    "distance"),
                 metric      = cfg.get("metric",     "euclidean"),
                 n_jobs=-1),
}

In [31]:
@tool
def read_data_tool(csv_path: str) -> dict:
    """
    Read CSV, identify appropriate target column automatically,
    return column names split into features and target.
    """
    df   = pd.read_csv(csv_path, nrows=5)
    cols = df.columns.tolist()
    date_cols    = [c for c in cols if any(k in c.lower()
                    for k in ["date", "caldt", "time", "period"])]
    numeric_cols = [c for c in cols if c not in date_cols]
 
    known_targets = ["cpiret", "gdp", "volatility", "unrate", "nfci"]
    target   = next((c for c in numeric_cols if c.lower() in known_targets),
                    numeric_cols[-1])
    features = [c for c in numeric_cols if c != target]
 
    return {
        "target_column":   target,
        "feature_columns": features,
        "date_columns":    date_cols,
        "all_columns":     cols
    }

In [47]:

read_data_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are a data inspection agent.
    Call read_data_tool with the CSV path provided.
    Return the result exactly as the tool returns it.
    """),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

read_data_agent = create_tool_calling_agent(
    llm,
    [read_data_tool],
    read_data_prompt
)

read_data_executor = AgentExecutor(
    agent=read_data_agent,
    tools=[read_data_tool],
    verbose=True,
    return_intermediate_steps=True
)

In [33]:
@tool
def preprocess_tool(df_path: str, target_col: str,
                    feature_cols: list, date_cols: list) -> dict:
    """
    Preprocess dataset: build lag/rolling features, split, save CSVs.
    """
    output_dir = "D:/mycode/Fintech-agent/data/processed_data"
    os.makedirs(output_dir, exist_ok=True)
 
    df = pd.read_csv(df_path)
 
    df = df.drop(columns=date_cols, errors="ignore")
    df = df[[c for c in feature_cols + [target_col] if c in df.columns]]
    df = df.dropna().reset_index(drop=True)

    for lag in [1, 2, 4]:
        df[f"{target_col}_lag{lag}"] = df[target_col].shift(lag)
        for col in feature_cols:
            if col in df.columns:
                df[f"{col}_lag{lag}"] = df[col].shift(lag)
 
    for w in [2, 4]:
        df[f"{target_col}_roll{w}"] = df[target_col].shift(1).rolling(w).mean()
 
    df = df.dropna().reset_index(drop=True)
 
    X = df.drop(columns=[target_col])
    y = df[target_col]
 
    X_train, X_test = X.iloc[:-8], X.iloc[-8:]
    y_train, y_test = y.iloc[:-8], y.iloc[-8:]
 
    paths = {
        "X_train": os.path.join(output_dir, "X_train.csv"),
        "X_test":  os.path.join(output_dir, "X_test.csv"),
        "y_train": os.path.join(output_dir, "y_train.csv"),
        "y_test":  os.path.join(output_dir, "y_test.csv"),
    }
    X_train.to_csv(paths["X_train"], index=False)
    X_test.to_csv( paths["X_test"],  index=False)
    y_train.to_csv(paths["y_train"], index=False)
    y_test.to_csv( paths["y_test"],  index=False)
 
    print(f"  Preprocessed: {len(X_train)} train, {len(X_test)} test, "
          f"{X_train.shape[1]} features (with lags/rolling)")
    return paths

In [34]:
preprocess_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a data preprocessing agent.
Call preprocess_tool with:
- df_path: the csv file path
- target_col: the target column
- feature_col: list of feature column names
- date_cols: list of date column names to drop

Return the file paths dictionary.
"""
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])


preprocess_agent = create_tool_calling_agent(
    llm,
    [preprocess_tool],
    preprocess_prompt
)


preprocess_executor = AgentExecutor(
    agent=preprocess_agent,
    tools=[preprocess_tool],
    verbose=True
)

In [35]:
def escape(text: str) -> str:
        #Escape all curly braces except LangChain placeholders.
        return text.replace("{", "{{").replace("}", "}}")

In [36]:
def forecast(data_paths, model_name, model_params):
    X_train = pd.read_csv(data_paths["X_train"])
    X_test = pd.read_csv(data_paths["X_test"])
    y_train = pd.read_csv(data_paths["y_train"]).squeeze()
    y_test = pd.read_csv(data_paths["y_test"]).squeeze()

    model = MODEL_REGISTRY[model_name](model_params)
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #calc SHAP
    if hasattr(model, "predict_proba") or "tree" in model_name or "forest" in model_name:
        explainer = shap.Explainer(model, X_train)
    else:
        explainer = shap.Explainer(model.predict, X_train)

    shap_values = explainer(X_test)

    shap_result = (
        pd.DataFrame({
            "feature": X_test.columns,
            "importance": abs(shap_values.values).mean(axis=0)
        })
        .sort_values(by="importance", ascending=False)
        .reset_index(drop=True)
        .to_dict(orient="records")
    )

    return {
        "model_name":  model_name,
        "model_params": model_params,
        "train_r2":    round(float(r2_score(y_train, y_train_pred)), 4),
        "test_r2":     round(float(r2_score(y_test,  y_test_pred)),  4),
        "test_rmse":   round(float(np.sqrt(mean_squared_error(y_test, y_test_pred))), 6),
        "test_mape":   round(float(mean_absolute_percentage_error(y_test, y_test_pred) * 100), 4),
        "shap_result": shap_result,
        "y_test": y_test.tolist(),
        "y_test_pred": y_test_pred.tolist()
    }

In [37]:
@tool
def forecast_tool(data_paths: dict, model_name: str, model_params: dict) -> dict:
    """
    Train a selected regression model using CSV paths and return evaluation metrics.
    model_name must be one of: bayes, gbr, xgb, svr, knn
    model_params: Required dict of hyperparameters
    """
    return forecast(data_paths, model_name=model_name, model_params=model_params)

In [38]:
def extract_suggested_params(llm_output: str) -> dict:
    matches = re.findall(r'\{[^{}]+\}', llm_output)
    for m in matches:
        try:
            parsed = json.loads(m)
            if any(isinstance(v, (int, float)) for v in parsed.values()):
                return parsed
        except Exception:
            pass
    return {}

In [ ]:

def build_forecast_prompt(history: list = []) -> ChatPromptTemplate:
    history_text      = ""
    forbidden_configs = []
 
    if history:
        history_text = "\n\nPREVIOUS ITERATION RESULTS:\n"
        for i, h in enumerate(history):
            cfg = h.get("model_params", {})
            forbidden_configs.append(cfg)
            history_text += (
                f"\nIteration {i+1}: model={escape(h['model_name'])} "
                f"train_r2={h['train_r2']} test_r2={h['test_r2']} "
                f"config={escape(json.dumps(cfg))}\n"
            )
 
        forbidden_str = escape(json.dumps(forbidden_configs))
        history_text += (
            "\nOPTIMIZATION INSTRUCTIONS:\n"
            "1. If train_r2 >> test_r2: overfitting — increase reg_lambda, reduce max_depth.\n"
            "2. If test_r2 < 0: switch model or drastically increase regularization.\n"
            "3. Always propose NEW model_params — never reuse forbidden configs.\n"
            f"FORBIDDEN CONFIGS (never reuse): {forbidden_str}\n"
        )
 
    system_msg = (
        "You are a forecasting agent.\n"
        "You MUST call forecast_tool with THREE arguments:\n"
        "  1. data_paths: dict with X_train, X_test, y_train, y_test\n"
        "  2. model_name: one of xgb, gbr, svr, bayes, knn\n"
        "  3. model_params: NON-EMPTY dict of hyperparameters\n\n"
        "RULES: never pass empty model_params.\n\n"
        f"HYPERPARAM_REFERENCE:\n{escape(HYPERPARAM_REFERENCE)}\n"
        f"{escape(history_text)}\n"
        "Call forecast_tool NOW."
    )
 
    return ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}")
    ])
 

### Optimization Loop

In [ ]:

def run_optimization_loop(csv_path: str, max_iter: int):
    pass
 
    return best_metrics, history         

In [ ]:
best_metrics, history = run_optimization_loop(
        csv_path = r"D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv",
        max_iter = 3
    )

In [48]:
read_result = read_data_executor.invoke({"input":'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv'})
read_result



> Entering new AgentExecutor chain...

Invoking: `read_data_tool` with `{'csv_path': 'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv'}`


{'target_column': 'cpiret', 'feature_columns': ['b30ret', 'b20ret', 'b10ret', 'b7ret', 'b5ret', 'b2ret', 'b1ret', 't90ret', 't30ret'], 'date_columns': ['caldt'], 'all_columns': ['caldt', 'b30ret', 'b20ret', 'b10ret', 'b7ret', 'b5ret', 'b2ret', 'b1ret', 't90ret', 't30ret', 'cpiret']}The data in the CSV file 'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv' has been successfully read. The tool has identified the following columns:

- Target column: cpiret
- Feature columns: b30ret, b20ret, b10ret, b7ret, b5ret, b2ret, b1ret, t90ret, t30ret
- Date columns: caldt
- All available columns: caldt, b30ret, b20ret, b10ret, b7ret, b5ret, b2ret, b1ret, t90ret, t30ret, cpiret

> Finished chain.


{'input': 'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv',
 'output': "The data in the CSV file 'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv' has been successfully read. The tool has identified the following columns:\n\n- Target column: cpiret\n- Feature columns: b30ret, b20ret, b10ret, b7ret, b5ret, b2ret, b1ret, t90ret, t30ret\n- Date columns: caldt\n- All available columns: caldt, b30ret, b20ret, b10ret, b7ret, b5ret, b2ret, b1ret, t90ret, t30ret, cpiret",
 'intermediate_steps': [(ToolAgentAction(tool='read_data_tool', tool_input={'csv_path': 'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv'}, log="\nInvoking: `read_data_tool` with `{'csv_path': 'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv'}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-05-05T08:34:59.331926Z', 'done': True, 'done_reason': 'stop', 'total_duration': 101012885900, 'l

In [54]:
column_info = read_result['intermediate_steps'][-1][1]
column_info

{'target_column': 'cpiret',
 'feature_columns': ['b30ret',
  'b20ret',
  'b10ret',
  'b7ret',
  'b5ret',
  'b2ret',
  'b1ret',
  't90ret',
  't30ret'],
 'date_columns': ['caldt'],
 'all_columns': ['caldt',
  'b30ret',
  'b20ret',
  'b10ret',
  'b7ret',
  'b5ret',
  'b2ret',
  'b1ret',
  't90ret',
  't30ret',
  'cpiret']}

In [55]:
preprocess_input = {
        "df_path": 'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv',
        "target_col": column_info.get("target_column"),
        "feature_cols": column_info.get("feature_columns"),
        "date_cols": column_info.get("date_columns",[]),
    }

In [56]:
preprocess_input["target_col"]

'cpiret'

In [15]:
df   = pd.read_csv("D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv", nrows=5)
cols = df.columns.tolist()
date_cols    = [c for c in cols if any(k in c.lower()
                for k in ["date", "caldt", "time", "period"])]
numeric_cols = [c for c in cols if c not in date_cols]

known_targets = ["cpiret", "gdp", "volatility", "unrate", "nfci"]
target   = next((c for c in numeric_cols if c.lower() in known_targets),
                numeric_cols[-1])
features = [c for c in numeric_cols if c != target]

In [ ]:
column_info = read_result["output"]

In [55]:
forecast_prompt = build_forecast_prompt()

forecast_agent = create_tool_calling_agent(
    llm,
    [forecast_tool],
    forecast_prompt
)


forecast_executor = AgentExecutor(
    agent=forecast_agent,
    tools=[forecast_tool],
    verbose=True,
    handle_parsing_errors=True,
    return_intermediate_steps=True
)

In [52]:
# Step 1 — Preprocess dataset
preprocess_result = preprocess_executor.invoke(
    {"input": "D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv"}
)

preprocessed_paths = preprocess_result["output"]
print(preprocessed_paths)



> Entering new AgentExecutor chain...

Invoking: `preprocess_tool` with `{'df_path': 'D:/mycode/Fintech-agent/data/macro/Treasury and Inflation.csv'}`


{'X_train': 'D:/mycode/Fintech-agent/data/processed_data\\X_train.csv', 'X_test': 'D:/mycode/Fintech-agent/data/processed_data\\X_test.csv', 'y_train': 'D:/mycode/Fintech-agent/data/processed_data\\y_train.csv', 'y_test': 'D:/mycode/Fintech-agent/data/processed_data\\y_test.csv'}The processed datasets have been saved as CSV files in the 'D:/mycode/Fintech-agent/data/processed_data' directory. The file paths for the training and testing datasets are:

- X_train: D:/mycode/Fintech-agent/data/processed_data/X_train.csv
- X_test: D:/mycode/Fintech-agent/data/processed_data/X_test.csv
- y_train: D:/mycode/Fintech-agent/data/processed_data/y_train.csv
- y_test: D:/mycode/Fintech-agent/data/processed_data/y_test.csv

> Finished chain.
The processed datasets have been saved as CSV files in the 'D:/mycode/Fintech-agent/data/processed_data' di

In [57]:
# Step 2 — Train model
forecast_result = forecast_executor.invoke(
    {"input": preprocessed_paths}
)

metrics = forecast_result["intermediate_steps"][-1][1]
print(metrics)



> Entering new AgentExecutor chain...

Invoking: `forecast_tool` with `{'data_paths': {'X_test': 'D:/mycode/Fintech-agent/data/processed_data/X_test.csv', 'X_train': 'D:/mycode/Fintech-agent/data/processed_data/X_train.csv', 'X_val': '', 'y_test': 'D:/mycode/Fintech-agent/data/processed_data/y_test.csv', 'y_train': 'D:/mycode/Fintech-agent/data/processed_data/y_train.csv', 'y_val': ''}, 'model_name': 'gbr', 'model_config': {}}`


{'model_name': 'gbr', 'model_config': {}, 'train_r2': 0.9549260079832782, 'test_r2': -0.26427585707597734, 'test_rmse': 0.007911871696219748, 'test_mape': 203.53638599943088, 'shap_result': [{'feature': 't30ret', 'importance': 0.0020532962431800057}, {'feature': 'b30ret', 'importance': 0.0018728170483806377}, {'feature': 'b5ret', 'importance': 0.0015923571220791467}, {'feature': 'b10ret', 'importance': 0.0009640464174085976}, {'feature': 'b20ret', 'importance': 0.0009266902342950607}, {'feature': 'b1ret', 'importance': 0.0007668167545039413}, {'feature': 'b2

### Report generation Agent

In [58]:
report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a machine learning report generator.

Given evaluation metrics, generate a professional Markdown report including:

- Model Overview
- Performance Table
- Interpretation of metrics
- Overfitting Assessment
- Conclusion
"""
    ),
    ("human", "{input}")
])

report_chain = report_prompt | llm

In [59]:
if "shap_importance" in metrics:
    metrics["shap_importance"] = dict(
        sorted(metrics["shap_importance"].items(), key=lambda x: x[1], reverse=True))
    

# report_input = {
#     "history":      history,
#     "best_metrics": best_metrics,
#     "iterations":   len(history)
# }
# Step 3 — Generate report
report = report_chain.invoke(
    {"input": metrics} #best_metrics/report_input/metrics
)

print(report)

content="**Model Overview**\n================\n\nThe model used for this evaluation is a Gradient Boosting Regressor (GBR) with default configuration.\n\n**Performance Table**\n-------------------\n\n| Metric | Train | Test |\n| --- | --- | --- |\n| R2 Score | 0.955 | -0.264 |\n| RMSE | N/A | 0.0079 |\n| MAPE | N/A | 203.54 |\n\nNote: The test R2 score is negative, indicating that the model has overfit to the training data.\n\n**Interpretation of Metrics**\n---------------------------\n\n*   **R2 Score**: Measures the proportion of variance in the target variable explained by the model. A higher value indicates better fit.\n    *   Train R2 score: 0.955 (very good)\n    *   Test R2 score: -0.264 (poor, indicating overfitting)\n*   **RMSE (Root Mean Squared Error)**: Measures the average magnitude of the errors made by the model. A lower value indicates better performance.\n    *   Test RMSE: 0.0079\n*   **MAPE (Mean Absolute Percentage Error)**: Measures the average absolute percentage

In [5]:
with open("result/forecast_report.md", "w", encoding="utf-8") as f:
    f.write(report["content"])

print("Report saved to forecast_report.md")


Report saved to forecast_report.md


### References
https://docs.langchain.com/oss/python/integrations/chat/ollama

https://docs.langchain.com/oss/python/langchain/tools


- feed more data about the forecasting process and feature engineering information for better report
- purpose of the task (is it to understand how agentic system works or should we try to build an efficient system for real life scenario)
- should the tool be called by the agent 
